# Bill Scanner - train the model (PHP 100/500/1000, USD 1/5/20, EUR 5/10/20)

Run the cells **from top to bottom** (Shift+Enter). Before you start:

1. **Runtime -> Change runtime type -> T4 GPU** (free).
2. Label your photos in Roboflow (or CVAT) with exactly these nine classes: `php_100`, `php_500`, `php_1000`, `usd_1`, `usd_5`, `usd_20`, `eur_5`, `eur_10`, `eur_20`.
3. Export the dataset as **YOLOv8** format (zip) and upload it to the root of your Google Drive as `bills.zip`.

At the end the notebook downloads `app_files.zip` containing `model.tflite` and `labels.txt`. Copy both into the Flutter app's `assets/` folder.

In [ ]:
!pip install -q ultralytics

## Step 1 - Load the dataset from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_ZIP = '/content/drive/MyDrive/bills.zip'   # <- change if your zip is elsewhere

!rm -rf /content/dataset && mkdir -p /content/dataset
!unzip -q "$DATASET_ZIP" -d /content/dataset
!ls /content/dataset

## Step 2 - Fix `data.yaml` (so the paths always work) and check the classes

In [ ]:
import yaml, pathlib

root = pathlib.Path('/content/dataset')
yaml_path = next(root.rglob('data.yaml'))
ds_root = yaml_path.parent
cfg = yaml.safe_load(yaml_path.read_text())

names = cfg['names']
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
print('Classes (in model order):', names)

def pick(*candidates):
    for c in candidates:
        if (ds_root / c / 'images').exists():
            return f'{c}/images'
    return None

fixed = {
    'path': str(ds_root),
    'train': pick('train'),
    'val': pick('valid', 'val'),
    'names': names,
}
test_dir = pick('test')
if test_dir:
    fixed['test'] = test_dir
assert fixed['train'] and fixed['val'], 'Could not find the train / valid image folders in the zip'

yaml_path.write_text(yaml.safe_dump(fixed, sort_keys=False))
DATA_YAML = str(yaml_path)

expected = {
    'php_100', 'php_500', 'php_1000',
    'usd_1', 'usd_5', 'usd_20',
    'eur_5', 'eur_10', 'eur_20',
}
unknown = sorted(set(names) - expected)
missing = sorted(expected - set(names))
if unknown:
    print('\nWARNING: these class names are not in the app\'s bill table:', unknown)
    print('The app still shows them as plain text. To get the nice names, rename the classes.')
else:
    print('Class names look good.')
    if missing:
        print('This model does not cover these bills yet:', missing)


## Step 3 - Train

Takes roughly 10-30 minutes on the free GPU, depending on dataset size. `patience=25` stops early when it stops improving.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')   # smallest YOLO11 model: fast enough for phones
model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=25,
    project='/content/runs',
    name='bills',
    exist_ok=True,
    # Bills are never mirrored in real life, and color is the main clue
    # between denominations, so: no flipping, only a tiny hue shift.
    fliplr=0.0,
    flipud=0.0,
    hsv_h=0.01,
    degrees=10,
)
BEST = str(model.trainer.best)
print('Best weights:', BEST)

## Step 4 - Check how good it is

Look at the numbers (mAP50 close to 1.0 is great) and the **confusion matrix**: it shows which denominations get mixed up. If two classes are confused, add more photos of them and train again.

In [ ]:
import os
from IPython.display import Image, display

best_model = YOLO(BEST)
split = 'test' if 'test' in fixed else 'val'
metrics = best_model.val(data=DATA_YAML, split=split)
print(f'mAP50: {metrics.box.map50:.3f}    mAP50-95: {metrics.box.map:.3f}    (evaluated on: {split})')

run_dir = pathlib.Path(BEST).parent.parent
cm = run_dir / 'confusion_matrix.png'
if cm.exists():
    display(Image(str(cm)))

### Quick visual check on a few photos it has never trained on

In [ ]:
import random

eval_dir = ds_root / (fixed.get('test') or fixed['val'])
images = sorted(p for p in eval_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
sample = random.sample(images, k=min(6, len(images)))

preds = best_model.predict(
    [str(p) for p in sample], conf=0.5, save=True,
    project='/content/preds', name='check', exist_ok=True,
)
for r in preds:
    display(Image(os.path.join(r.save_dir, os.path.basename(r.path))))

## Step 5 - Export for the phone and download

Creates `model.tflite` (float16) and `labels.txt` (generated from the model itself, so the order is always right).

In [ ]:
import glob, shutil

YOLO(BEST).export(format='tflite', imgsz=640)

weights_dir = os.path.dirname(BEST)
found = glob.glob(os.path.join(weights_dir, '**', '*_float16.tflite'), recursive=True)
assert found, 'No float16 .tflite file was created - scroll up for the export error and re-run this cell.'

out = pathlib.Path('/content/app_files')
shutil.rmtree(out, ignore_errors=True)
out.mkdir()
shutil.copy(found[0], out / 'model.tflite')

class_names = YOLO(BEST).names          # {0: 'php_100', 1: ..., ...}
(out / 'labels.txt').write_text('\n'.join(class_names[i] for i in range(len(class_names))) + '\n')
print('labels.txt:')
print((out / 'labels.txt').read_text())

shutil.make_archive('/content/app_files', 'zip', out)

from google.colab import files
files.download('/content/app_files.zip')

## Done

Unzip `app_files.zip` and copy **both** `model.tflite` and `labels.txt` into `bill_scanner/assets/`, then run the app (README.md, step 4).